[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/pose_diffusion/station_keeping.ipynb)

# Station Keeping in Gusts

This example computes how far a vessel on dynamic positioning wanders from its set point. The vessel holds a fixed spot with its thrusters while wind and waves push it around, so it never sits exactly on its set point: it wanders, and more sideways than forward, because gusts hit the side of the hull harder than the bow. Its spread says how much clear water the vessel needs around it and how stiff the thruster controller has to be; it is predicted directly, checked against a simulated fleet, and used to choose a controller gain.

The vessel's error is modelled as a twist, the small motion from the set point to where the vessel is, and the spread of the errors as a covariance, a map from lines to twists: a line measures an error, `line & error`, and `line & covariance(line)` is that measurement's variance. The controller acts on the covariance from both sides, on the twists it returns and, through `controller.adjugate()`, on the lines it takes, and the steady spread is one solve, `balance.lstsq(-gust_covariance)`, batched over every controller gain at once.

In matrix notation the error reads as a coordinate vector $(v_x, v_y, \omega)$ in a chosen order, the covariance as a symmetric $3\times3$ matrix $P$ with $\dot P = F P + P F^\top + Q$, and the steady spread as the solution of the continuous Lyapunov equation $F P + P F^\top + Q = 0$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.animation import save_figure
from examples.estimation.pose_diffusion import render

np.set_printoptions(precision=3, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Twist = ga.gatype.bivector()            # a small motion: a rotation and a translation together
Line = ga.gatype.antibivector()         # a line measures a twist: line & twist is a number
Covariance = ga.gatype((Twist, Line))   # Twist <- Line
Dynamics = ga.gatype((Twist, Twist))    # Twist <- Twist
set_point = mv.w.dual()                 # the set point, at the origin; in PGA2D a point is a bivector
print(Twist, Line, sep="\n")

## 1. Gusts and the controller

The vessel's error is a twist: the small motion that takes the set point to where the vessel actually is. Gusts push it at random. A gust is modelled as white noise sent through the gust map, a map from lines to twists, a sum of dyads over the directions gusts push in, sideways, forward and in yaw, each at its own strength. The controller is a map on twists. It reduces the error by a fixed fraction per second, the gain, so the error decays with a time constant of one over the gain. If the vessel also turns on the spot, its error turns with it; the commutator with the turning rate adds that.

In matrix notation one chooses an order for the twist's coordinates, say $(v_x, v_y, \omega)$, and writes the dynamics as $F = \operatorname{ad}_\Omega - k\,I$, with $\operatorname{ad}_\Omega$ the $3\times3$ matrix of the commutator with the turning rate in that order, signs included, and the gust map as a $3\times3$ matrix $B$.

In [ ]:
# The directions gusts push in, in the vessel's frame, sideways, forward and in yaw, and how hard,
# per square root of a second: metres, metres and radians. The side of the hull catches much more than the bow.
pushes = stack([mv.yw, mv.wx, mv.xy])                                         # [pushes] Twist
strengths = np.array([0.5, 0.15, 0.01])                                       # [pushes]
# The controller's gain per second: a 10 s time constant.
gain = 0.1

In [ ]:
# A gust: each push direction at its strength, read off the white noise by the line it pairs with.
gusts = (pushes * strengths * (pushes & Line)).sum(axis=-1)                   # Twist <- Line


def drift(turning: Twist, gain: Scalar) -> Dynamics:
    """How a position error changes per second: it turns with the vessel, and the controller shrinks it."""
    return Twist.commutator(turning) - gain * Twist                            # Twist <- Twist


still = mv.xy * 0.0                                                           # the vessel keeps its heading
controller = drift(still, mv.scalar([gain]))                                  # Twist <- Twist

## 2. The spread as a covariance

The spread of the vessel's position is a covariance. Here it is a map from lines to twists. A line measures the error, `line & error` is one number, and the covariance returns the twist that is correlated with that measurement; `line & covariance(line)` is the variance of the measurement. White noise is defined in a frame of lines, with unit variance along each and none between them, so the covariance the gusts add per second is the sum over that frame of each line's gust joined with itself.

In [ ]:
# The frame the white noise is sampled in: unit variance along each line, and none between them.
frame = mv.antibivector(np.eye(3))                                            # [3] Line
gust_covariance = (gusts(frame) * (gusts(frame) & Line)).sum(axis=0)          # Twist <- Line, per second

As the errors change, so does their covariance. The controller acts on the twists the covariance returns. It also changes which errors a given line picks out: measuring the changed error `controller(t)` with a line gives the same number as measuring `t` itself with another line. That other line is the controller's adjugate applied to the first, `controller.adjugate()`, the map on lines that incidence carries over from the controller. The covariance's rate of change is the sum of these two effects and the gusts.

In matrix notation the covariance is a symmetric $3\times3$ matrix $P$ and its rate of change is $\dot P = F P + P F^\top + Q$, with $Q = B B^\top$, and the transpose $F^\top$ reads as `dynamics.adjugate()`, the controller acting on measurements.

In [ ]:
def growth(dynamics: Dynamics, covariance: Covariance) -> Covariance:
    """Rate of change of the covariance: the dynamics on the twists it returns and on the lines it
    takes, plus what the gusts add."""
    return dynamics(covariance) + covariance(dynamics.adjugate()) + gust_covariance

## 3. The steady state in one solve

After a while the controller removes spread as fast as the gusts add it, and the covariance stops growing. The growth is linear in the covariance, so this balance is a linear equation whose unknown is a map. Written as a sum of dyads, a twist times a measurement of another twist, the covariance makes the growth a map of three inputs, the two twists and the line, `balance`. `lstsq` matches its line input against the gusts and solves for the other two at once. The result takes a line through its dual, the twist that measures the same as the line.

In matrix notation this is the continuous Lyapunov equation $F P + P F^\top + Q = 0$. It is solved either by vectorizing, $(I \otimes F + F \otimes I)\,\operatorname{vec} P = -\operatorname{vec} Q$, a $9\times9$ system whose matrix is `balance` with its two twist inputs flattened into one, or by the Bartels–Stewart algorithm, `scipy.linalg.solve_continuous_lyapunov`.

In [ ]:
def settled(dynamics: Dynamics) -> Covariance:
    """The covariance at which growth stops."""
    balance = dynamics(Twist) * (Twist & Line) + Twist * (dynamics(Twist) & Line)   # Twist <- (Twist, Twist, Line)
    return balance.lstsq(-gust_covariance)(Line.dual())                            # Twist <- Line


steady = settled(controller)                                             # Twist <- Line

## 4. Checking against a fleet

A simulated fleet checks the prediction. In each step every vessel's error changes under the controller and takes its own gust, and the predicted covariance is advanced by its growth over the same step. White noise over a step has a spread proportional to the square root of the step's length, so the gusts are scaled by the square root of dt. The figure shows the vessels, each as a short arrow along its heading, with the integrated prediction (solid) and the steady state from the solve (dashed) as 2σ ellipses of position.

In [ ]:
n_vessels = 2000
seconds, dt = 150.0, 0.2
seed = 0
extent = 5.0                                                                  # m, half the plot's width

In [ ]:
def simulate(dynamics: Dynamics, n_vessels: int, seconds: float, dt: float, seed: int) -> tuple[Twist, Covariance]:
    """A simulated fleet and the predicted covariance of its errors, from all vessels on the set point."""
    rng = np.random.default_rng(seed)
    errors = mv.bivector(np.zeros((n_vessels, 3)))                        # [vessels] Twist: all on the set point
    predicted = gust_covariance * 0.0                                             # no spread at the start
    for _ in range(int(seconds / dt)):
        white = mv.antibivector(rng.normal(size=(n_vessels, 3))) * np.sqrt(dt)  # [vessels] Line: white noise over the step
        errors = errors + dynamics(errors) * dt + gusts(white)
        predicted = predicted + growth(dynamics, predicted) * dt
    return errors, predicted


errors, predicted = simulate(controller, n_vessels, seconds, dt, seed)
poses = (errors * 0.5).exp()                                           # [vessels] Motor: each vessel's pose
save_figure(render.draw_settled({"holding heading": (poses, predicted, steady)}, extent), "station_keeping");

## 5. Choosing the controller gain

A stiffer controller keeps the vessel closer, at the cost of more thruster work. Because the steady state is a single solve, it can be computed for a whole range of gains at once: the gain becomes a batch of scalars, and every map above broadcasts over it. The position's spread then follows by measuring how an error moves the set point. With a requirement that the vessel keep its 2σ excursion within 2 m in every direction, the plot reads off the smallest gain that does it.

In matrix notation the position covariance is $\Sigma = J P J^\top$, with $J$ the $2\times3$ Jacobian of the set point's displacement with respect to the twist coordinates; here, with the set point on the rotation centre, it picks out the two translation coordinates.

In [ ]:
gains = np.linspace(0.02, 0.5, 60)                                            # per second
tolerance = 2.0                                                               # m, the 2σ excursion allowed

In [ ]:
def position(covariance: Covariance):
    """Covariance of position measurements: a line measures how an error moves the set point."""
    moves = Twist.commutator(set_point)                                 # Point <- Twist: how an error moves the set point
    through = moves.adjugate()                                         # Line <- Line: a position measurement as a twist measurement
    return through & covariance(through)                               # Scalar <- (Line, Line)


spreads = position(settled(drift(still, mv.scalar(gains[:, None]))))   # [gains] Scalar <- (Line, Line)
save_figure(render.draw_envelope(gains, spreads, tolerance), "station_keeping_gain");

Along the vessel's side the 2σ excursion is the gust strength over the square root of twice the gain, so the 2 m tolerance needs a gain of 0.125 per second, an 8 s time constant. Forward, where gusts push less, the same gain leaves a lot of margin.

## 6. Turning on the spot

A vessel that turns while it holds position carries its error around with it. A sideways push becomes a forward error a quarter turn later, and gusts push less in that direction. The commutator with the turning rate adds exactly this to the dynamics, and nothing else in the calculation changes. At one turn a minute the spread comes out rounder and a little smaller along its long axis.

In [ ]:
turning_rate = mv.xy * (2 * np.pi / 60)                                       # one full turn a minute
turning_seed = 1

In [ ]:
turning = drift(turning_rate, mv.scalar([gain]))                              # Twist <- Twist
turning_errors, turning_predicted = simulate(turning, n_vessels, seconds, dt, turning_seed)
save_figure(render.draw_settled({"holding heading": (poses, predicted, steady),
                                 "one turn a minute": ((turning_errors * 0.5).exp(), turning_predicted, settled(turning))}, extent),
            "station_keeping_turning");

In [ ]:
# checks
# The steady state does not grow, the integrated prediction reaches it, and the fleet matches it.
np.testing.assert_allclose(growth(controller, steady).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(predicted.kernel, steady.kernel, atol=1e-6)
sample = (errors * (errors & Line)).mean(axis=0)                       # Twist <- Line
np.testing.assert_allclose(sample.kernel, steady.kernel, atol=0.1 * np.abs(steady.kernel).max())
# Sideways the variance is the gust strength squared over twice the gain.
np.testing.assert_allclose(position(steady).eigvalsh().to_array()[-1], strengths[0]**2 / (2 * gain), rtol=1e-8)